In [1]:
# Significant similarity to audio descriptions. 

import numpy as np
import pandas as pd

scores = np.load('scores/S1/imagined_speech/alpha_repeat-1.npz', allow_pickle=True)
print(scores.files)
score_names = ['window_scores', 'window_zscores', 'story_scores', 'story_zscores']
score_files = {}
for name in score_names:
    score_files[name] = scores[name].item()

print(score_files['window_zscores'])

['window_scores', 'window_zscores', 'story_scores', 'story_zscores']
{('alpha_repeat-1', 'WER'): array([ 1.80435178,  0.39458176,  0.7553415 ,  0.65794351,  1.1450376 ,
        0.40872733,  1.1891742 ,  1.95762043,  0.8796883 ,  1.39007397,
        1.05956781,  1.03223887,  0.67617003,  1.04232344,  0.7667776 ,
        1.21165486,  1.6068432 ,  1.7444016 ,  2.3785895 ,  1.65056143,
        1.02012571,  1.24973367,  0.87942521,  0.88902428,  1.09511849,
        0.2549425 ,  0.29451819, -0.83297723,  0.14611468,  1.01485667,
        0.40168006,  0.42833926,  0.27511051,  1.47019011,  1.73797237,
        2.00656433,  1.32424345,  1.44767558,  0.59966694,  0.72514339,
        1.56436002]), ('alpha_repeat-1', 'BLEU'): array([ 0.10711633,  0.14219169,  0.25286393,  0.54276823,  0.53881591,
        0.75968704,  0.67347769,  0.73676474,  0.81378788,  0.93893092,
        1.28485223,  1.2111761 ,  1.66839135,  2.04652061,  2.47520212,
        2.3985014 ,  1.92157021,  1.09932218,  0.75564003,  0

In [2]:
window_zscores = {'subject': [], 'WER':[],'BLEU':[], 'METEOR':[], 'BERT':[]}
for subject in [1,2,3]:
    for task in ["echo_repeat-2", "echo_repeat-1", "delta_repeat-2", "delta_repeat-1", "charlie_repeat-2", "charlie_repeat-1", "bravo_repeat-2", "bravo_repeat-1", "alpha_repeat-2", "alpha_repeat-1"]:
        scores = np.load(f'scores/S{subject}/imagined_speech/{task}.npz', allow_pickle=True)['window_zscores'].item()
        # print(scores['window_zscores'].item())
        window_zscores['subject'].append(subject)
        window_zscores['WER'].append(scores[(task, 'WER')])
        window_zscores['BLEU'].append(scores[(task, 'BLEU')])
        window_zscores['METEOR'].append(scores[(task, 'METEOR')])
        window_zscores['BERT'].append(scores[(task, 'BERT')])

window_zscores

{'subject': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3],
 'WER': [array([ 0.75809125,  0.79325623, -0.38108186,  0.58893509,  0.42379592,
          0.12406125,  0.12668014,  0.21300967,  0.39652579,  0.88379864,
         -1.02663737, -0.87235674, -0.73184869,  1.03087014,  0.81566965,
          0.43685203,  1.37047865,  2.25404901,  1.5539527 ,  1.63681681,
          1.76310782,  0.77937675,  1.46091528,  0.7916245 ,  0.9218049 ,
          2.03109225,  1.29076687,  1.10419199,  1.15790496,  0.72070552,
          2.11493191,  2.54107987,  1.15217391,  2.18012868,  1.10130242,
          1.17232895,  2.46773126,  2.33486893,  2.1384111 ,  1.80794315,
          2.38468063]),
  array([-0.28636438, -0.00595502,  1.03746722, -0.3074594 , -0.48154341,
          0.52470632,  0.4975186 ,  0.81660563,  1.53510742,  0.40080519,
          0.77701682,  0.76056299,  1.28499688,  0.39820018,  1.188

In [3]:
results_df = pd.DataFrame(window_zscores)

print(len(results_df.loc[0, 'WER']))
print(len(results_df.loc[1, 'WER']))
print(len(results_df.loc[2, 'WER']))
print(len(results_df.loc[3, 'WER']))
print(len(results_df.loc[4, 'WER']))
print(len(results_df.loc[5, 'WER']))
print(len(results_df.loc[6, 'WER']))
print(len(results_df.loc[7, 'WER']))
print(len(results_df.loc[8, 'WER']))
print(len(results_df.loc[9, 'WER']))
print('=')
print(41*10)
m=410

from scipy.stats import norm
#convert to p val
results_df['WER'] = results_df['WER'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['BLEU'] = results_df['BLEU'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['METEOR'] = results_df['METEOR'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['BERT'] = results_df['BERT'].apply(lambda l : [1-norm.cdf(z) for z in l])
#sort
for i in range(3):
    for met in ['WER','BLEU','METEOR','BERT']:
        results_df.loc[i,met].sort()
# to q and check threshold
def p_to_q(p, i):
    return p*m/i
results_df['WER'] = results_df['WER'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['BLEU'] = results_df['BLEU'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['METEOR'] = results_df['METEOR'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['BERT'] = results_df['BERT'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])

S1=np.array(results_df.loc[0, 'BERT'] + results_df.loc[1, 'BERT'] + results_df.loc[2, 'BERT'] + results_df.loc[3, 'BERT'] + results_df.loc[4, 'BERT'] + results_df.loc[5, 'BERT'] + results_df.loc[6, 'BERT'] + results_df.loc[7, 'BERT'] + results_df.loc[8, 'BERT'] + results_df.loc[9, 'BERT']).mean()
S2=np.array(results_df.loc[10, 'BERT'] + results_df.loc[11, 'BERT'] + results_df.loc[12, 'BERT'] + results_df.loc[13, 'BERT'] + results_df.loc[14, 'BERT'] + results_df.loc[15, 'BERT'] + results_df.loc[16, 'BERT'] + results_df.loc[17, 'BERT'] + results_df.loc[18, 'BERT'] + results_df.loc[19, 'BERT']).mean()
S3=np.array(results_df.loc[20, 'BERT'] + results_df.loc[21, 'BERT'] + results_df.loc[22, 'BERT'] + results_df.loc[23, 'BERT'] + results_df.loc[24, 'BERT'] + results_df.loc[25, 'BERT'] + results_df.loc[26, 'BERT'] + results_df.loc[27, 'BERT'] + results_df.loc[28, 'BERT'] + results_df.loc[29, 'BERT']).mean()
print(S1,S2,S3,'BERT')

results_df

41
41
41
41
41
41
41
41
41
41
=
410
0.08536585365853659 0.3902439024390244 0.20243902439024392 BERT


,subject,WER,BLEU,METEOR,BERT
0,1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,1,"[0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, ..."
4,1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
5,1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
6,1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
7,1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
8,1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
9,1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [5]:
to_file = pd.DataFrame({'subject':[1,2,3], 'significantly_decoded': [S1,S2,S3]})
to_file.to_csv('imagined_speech_percentages.csv', index=False)

to_file

,subject,significantly_decoded
0,1,0.085366
1,2,0.390244
2,3,0.202439


In [6]:
results = np.load('results/S1/perceived_speech/wheretheressmoke.npz', allow_pickle=True)
result_names = results.files
result_files={}
for name in result_names:
    result_files[name] = results[name]
result_files

{'words': array(['she', 'said', 'she', ..., 'that', 'she', 'needs'],
       shape=(1589,), dtype='<U13'),
 'times': array([ 10.2,  10.6,  11. , ..., 591. , 591.4, 591.8], shape=(1589,))}